# Qui uso il Firms NASA per fare la previsione di incendi sul territorio

## API per vedere la mia MAP_KEY per accedere ai dati via richieste API

In [ ]:
# # Let's set your map key that was emailed to you. It should look something like 'abcdef1234567890abcdef1234567890'
# MAP_KEY = '<replace with your map_key>'
MAP_KEY = '2c5ba4423dded18d0b41241170c6d284'

# now let's check how many transactions we have
import pandas as pd
import requests
url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  response = requests.get(url)
  data = response.json()
  df = pd.Series(data)
  display(df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

## API per vedere i datasets

In [ ]:
# let's query data_availability to find out what date range is available for various datasets
# we will explain these datasets a bit later

# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
df_datasets = pd.read_csv(da_url)
display(df_datasets)



## Script per salvare tutti i dataset per iterare sopra

In [ ]:
datasets = []
for i in df_datasets["data_id"]:
    datasets.append(i)
print(datasets)

## Data Ingest 

In [ ]:
dfs = []

for dataset in datasets:
    area_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{dataset}/world/1"
    df = pd.read_csv(area_url)
    df["dataset"] = dataset
    dfs.append(df)

df_finale = pd.concat(dfs, ignore_index=True)

In [ ]:
df_finale.shape

In [ ]:
df = df_finale.to_csv("dataset_incendi_FIRMS.csv", index=False)

In [ ]:
# Bounding box Italia
df_italia = df_finale[
    (df_finale['longitude'] >= 6) &
    (df_finale['longitude'] <= 19) &
    (df_finale['latitude'] >= 36) &
    (df_finale['latitude'] <= 47)
].copy()

print(df_italia)

In [ ]:
df_italia.shape

In [ ]:
import pandas as pd

df = pd.read_csv("dataset_incendi_FIRMS.csv") 

In [ ]:
df_italia = df_finale[
    (df_finale['longitude'] >= 6) &
    (df_finale['longitude'] <= 19) &
    (df_finale['latitude'] >= 36) &
    (df_finale['latitude'] <= 47)
].copy()

In [ ]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
)

In [ ]:
gdf.plot(column='brightness', figsize=(10, 6), legend=True, markersize=2)

In [ ]:
from IPython.display import display

m = gdf.explore(column="brightness", legend=True)